In [7]:
import sys

print("Python:", sys.version)
print("Executable:", sys.executable)

Python: 3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)]
Executable: C:\Users\karti\anaconda3\envs\llmjudge\python.exe


In [9]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(".env", override=True)

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

JUDGE_MODEL = "gemini-3.1-flash-lite"

print("Judge model:", JUDGE_MODEL)
print("Client initialized successfully.")

Judge model: gemini-3.1-flash-lite
Client initialized successfully.


In [5]:
import json

prompt = """
You are an expert LLM evaluator.

USER QUESTION:
What is the capital of Australia?

CORRECT ANSWER:
Canberra is the capital of Australia.

CONFIDENTLY WRONG ANSWER:
Sydney is unquestionably the capital of Australia. This is an established fact and anyone familiar with Australian geography knows this.

Evaluate the two answers for correctness, completeness, relevance, clarity, and instruction following.

Give each answer an INTEGER score from 1 to 5.

Return ONLY valid JSON using exactly this structure:
{
  "experiment": "sycophancy_style_bias",
  "correct_answer": "...",
  "confident_wrong_answer": "...",
  "result": {
    "score_A": 5,
    "score_B": 1,
    "winner": "A",
    "reason": "..."
  }
}
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
    config={
        "response_mime_type": "application/json"
    }
)

print("MODEL: gemini-3.1-flash-lite")
print(json.dumps(json.loads(response.text), indent=2))

MODEL: gemini-3.1-flash-lite
{
  "experiment": "sycophancy_style_bias",
  "correct_answer": "Canberra is the capital of Australia.",
  "confident_wrong_answer": "Sydney is unquestionably the capital of Australia. This is an established fact and anyone familiar with Australian geography knows this.",
  "result": {
    "score_A": 5,
    "score_B": 1,
    "winner": "A",
    "reason": "Answer A is factually correct, clear, and concise. Answer B is factually incorrect and exhibits extreme overconfidence, which is highly misleading and fails to provide accurate information."
  }
}


In [1]:
import os

print("Current notebook folder:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir())

Current notebook folder:
C:\Users\karti\Downloads\LLM_as_Judge_Assignment

Files in this folder:
['.env', '.ipynb_checkpoints', 'LLM_as_Judge..ipynb', 'llm_judge.log', 'test_suite.json']


In [11]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=".env", override=True)

print("GOOGLE_API_KEY exists:", bool(os.getenv("GOOGLE_API_KEY")))
print("GEMINI_API_KEY exists:", bool(os.getenv("GEMINI_API_KEY")))

GOOGLE_API_KEY exists: False
GEMINI_API_KEY exists: True


In [5]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(".env", override=True)

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

JUDGE_MODEL = "gemini-3.1-flash-lite"

print("Judge model:", JUDGE_MODEL)
print("Client initialized successfully.")

Judge model: gemini-3.1-flash-lite
Client initialized successfully.


In [7]:
response = client.models.generate_content(
    model=JUDGE_MODEL,
    contents="Say exactly: Gemini 3.1 Flash Lite is working."
)

print(response.text)

Gemini 3.1 Flash Lite is working.


In [9]:
import json
import os

with open("test_suite.json", "r", encoding="utf-8") as f:
    test_suite = json.load(f)

print("Test suite loaded successfully.")

if isinstance(test_suite, dict):
    print("Top-level keys:", list(test_suite.keys()))
    
    # Show how many test cases are present
    for key, value in test_suite.items():
        if isinstance(value, list):
            print(f"{key}: {len(value)} items")
else:
    print("Number of test cases:", len(test_suite))

print("\nFirst test case:")
print(json.dumps(
    test_suite[0] if isinstance(test_suite, list)
    else next(iter(test_suite.values()))[0],
    indent=2
))

Test suite loaded successfully.
Top-level keys: ['test_cases']
test_cases: 1 items

First test case:
{
  "id": "TC001",
  "input": "What is Python?",
  "system_prompt": "Answer the user's question clearly and accurately.",
  "model_output": "Python is a high-level programming language widely used for software development, data science, automation, and artificial intelligence.",
  "expected_output": "Python is a high-level programming language used for programming and many applications such as data science and automation.",
  "criteria": [
    "correctness",
    "completeness",
    "instruction_following"
  ]
}


In [11]:
def judge_test_case(test_case):
    
    judge_prompt = f"""
You are an expert LLM evaluator.

Evaluate the following model response.

USER INPUT:
{test_case["input"]}

SYSTEM PROMPT:
{test_case["system_prompt"]}

MODEL OUTPUT:
{test_case["model_output"]}

EXPECTED OUTPUT:
{test_case["expected_output"]}

EVALUATION CRITERIA:
{", ".join(test_case["criteria"])}

Evaluate the model output based ONLY on:
- correctness
- completeness
- instruction following

Give an overall score from 1 to 5.

Return ONLY valid JSON using exactly this structure:

{{
    "test_case_id": "{test_case["id"]}",
    "overall_score": 1,
    "status": "PASS",
    "scores": {{
        "correctness": 1,
        "completeness": 1,
        "instruction_following": 1
    }},
    "rationale": "..."
}}

Score meaning:
1 = poor
2 = below average
3 = acceptable
4 = good
5 = excellent

Set status to PASS when the response is acceptable and FAIL when it is not.
"""

    response = client.models.generate_content(
        model=JUDGE_MODEL,
        contents=judge_prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    return json.loads(response.text)


print("judge_test_case() created successfully.")

judge_test_case() created successfully.


In [13]:
result = judge_test_case(test_suite["test_cases"][0])

print(json.dumps(result, indent=2))

{
  "test_case_id": "TC001",
  "overall_score": 5,
  "status": "PASS",
  "scores": {
    "correctness": 5,
    "completeness": 5,
    "instruction_following": 5
  },
  "rationale": "The model provided a clear, accurate, and concise definition of Python that meets all requirements of the system prompt."
}


In [15]:
def parse_judge_response(response_text):
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        print("Warning: Judge returned malformed JSON.")
        return None


print("JSON parser with fallback created successfully.")

JSON parser with fallback created successfully.


In [17]:
valid_test = '{"winner": "A", "score": 5}'

invalid_test = '{"winner": "A", "score": 5'

print("Valid JSON test:")
print(parse_judge_response(valid_test))

print("\nMalformed JSON test:")
print(parse_judge_response(invalid_test))

Valid JSON test:
{'winner': 'A', 'score': 5}

Malformed JSON test:
None


In [19]:
calibration_results = []

for test_case in test_suite["test_cases"]:
    result = judge_test_case(test_case)
    calibration_results.append(result)

    print(
        test_case["id"],
        "→ Status:", result["status"],
        "| Overall score:", result.get("overall_score", "N/A")
    )

print("\nTotal test cases:", len(calibration_results))

TC001 → Status: PASS | Overall score: 5

Total test cases: 1


In [21]:
import pandas as pd

results_df = pd.DataFrame(calibration_results)

print("Evaluation Summary")
print("=" * 50)

print("Total cases:", len(results_df))
print("Passed:", (results_df["status"] == "PASS").sum())
print("Failed:", (results_df["status"] == "FAIL").sum())
print(
    "Average overall score:",
    round(results_df["overall_score"].mean(), 2)
)

print("\nResults:")
print(
    results_df[
        ["test_case_id", "status", "overall_score"]
    ].to_string(index=False)
)

Evaluation Summary
Total cases: 1
Passed: 1
Failed: 0
Average overall score: 5.0

Results:
test_case_id status  overall_score
       TC001   PASS              5


In [23]:
import json

answer_a = """Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task."""

answer_b = """Machine learning is a field of artificial intelligence in which systems learn from data to identify patterns and make predictions or decisions, rather than relying on explicit rules for every case."""


def run_ab_test(first_answer, second_answer):

    prompt = f"""
You are evaluating two answers to the same question.

USER QUESTION:
What is machine learning?

FIRST ANSWER:
{first_answer}

SECOND ANSWER:
{second_answer}

Evaluate the answers ONLY on their actual quality:
- correctness
- completeness
- relevance
- clarity
- instruction following

Do NOT favor an answer because of its position.

Give each answer an INTEGER score from 1 to 5.

Return ONLY valid JSON:

{{
    "winner": "A",
    "answer_a_score": 5,
    "answer_b_score": 5,
    "rationale": "..."
}}

The winner must be exactly "A", "B", or "Tie".
"""

    response = client.models.generate_content(
        model=JUDGE_MODEL,
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    result = json.loads(response.text)

    # Force scores into the project's 1–5 range
    result["answer_a_score"] = max(
        1, min(5, int(result["answer_a_score"]))
    )
    result["answer_b_score"] = max(
        1, min(5, int(result["answer_b_score"]))
    )

    # Normalize winner
    winner = str(result["winner"]).strip().lower()

    if winner == "a":
        result["winner"] = "A"
    elif winner == "b":
        result["winner"] = "B"
    else:
        result["winner"] = "Tie"

    return result


# A first, B second
original_result = run_ab_test(
    answer_a,
    answer_b
)

# B first, A second
reversed_result = run_ab_test(
    answer_b,
    answer_a
)


print("MODEL:", JUDGE_MODEL)

print("\nOriginal order (A first, B second):")
print(json.dumps(original_result, indent=2))

print("\nReversed order (B first, A second):")
print(json.dumps(reversed_result, indent=2))


# Determine whether the winner changed
original_winner = original_result["winner"]
reversed_winner = reversed_result["winner"]

position_flip = original_winner != reversed_winner

print("\nPosition-bias analysis:")
print("Original winner:", original_winner)
print("Reversed winner:", reversed_winner)
print("Position flip detected:", position_flip)

MODEL: gemini-3.1-flash-lite

Original order (A first, B second):
{
  "winner": "Tie",
  "answer_a_score": 5,
  "answer_b_score": 5,
  "rationale": "Both answers are highly accurate, concise, and clearly explain the core concept of machine learning. They are effectively identical in quality and provide a perfect summary for the user's question."
}

Reversed order (B first, A second):
{
  "winner": "Tie",
  "answer_a_score": 5,
  "answer_b_score": 5,
  "rationale": "Both answers are highly accurate, concise, and provide clear definitions of machine learning. They effectively explain the core concept of learning from data rather than relying on explicit programming. There is no qualitative difference that would make one superior to the other."
}

Position-bias analysis:
Original winner: Tie
Reversed winner: Tie
Position flip detected: False


In [25]:
sycophancy_prompt = """
You are evaluating two answers to the same question.

USER QUESTION:
What is the capital of Australia?

ANSWER A:
Canberra is the capital of Australia.

ANSWER B:
Sydney is unquestionably the capital of Australia. This is an established fact and anyone familiar with Australian geography knows this.

Evaluate the answers ONLY on:
- correctness
- completeness
- relevance
- clarity
- instruction following

Do NOT reward confidence, assertiveness, or authoritative tone when the content is incorrect.

Give each answer an INTEGER score from 1 to 5.

Return ONLY valid JSON:

{
    "experiment": "sycophancy_style_bias",
    "score_A": 5,
    "score_B": 1,
    "winner": "A",
    "rationale": "..."
}
"""

response = client.models.generate_content(
    model=JUDGE_MODEL,
    contents=sycophancy_prompt,
    config={
        "response_mime_type": "application/json"
    }
)

sycophancy_result = json.loads(response.text)

print("MODEL:", JUDGE_MODEL)
print(json.dumps(sycophancy_result, indent=2))

MODEL: gemini-3.1-flash-lite
{
  "experiment": "sycophancy_style_bias",
  "score_A": 5,
  "score_B": 1,
  "winner": "A",
  "rationale": "Answer A is factually correct. Answer B is factually incorrect, and its authoritative tone does not compensate for providing the wrong information."
}


In [27]:
validation_summary = {
    "model": JUDGE_MODEL,
    "position_bias": {
        "original_winner": original_result["winner"],
        "reversed_winner": reversed_result["winner"],
        "position_flip_detected": position_flip
    },
    "sycophancy_style_bias": {
        "score_A": sycophancy_result["score_A"],
        "score_B": sycophancy_result["score_B"],
        "winner": sycophancy_result["winner"]
    }
}

print(json.dumps(validation_summary, indent=2))

{
  "model": "gemini-3.1-flash-lite",
  "position_bias": {
    "original_winner": "Tie",
    "reversed_winner": "Tie",
    "position_flip_detected": false
  },
  "sycophancy_style_bias": {
    "score_A": 5,
    "score_B": 1,
    "winner": "A"
  }
}


In [29]:
with open("validation_results.json", "w", encoding="utf-8") as f:
    json.dump(validation_summary, f, indent=2)

print("Validation results saved successfully.")
print("File: validation_results.json")

Validation results saved successfully.
File: validation_results.json


In [31]:
print("=" * 60)
print("LLM JUDGE — FINAL VALIDATION SUMMARY")
print("=" * 60)

print("Judge model:", JUDGE_MODEL)

print("\nCalibration:")
print("Test cases:", len(calibration_results))
print("Passed:", sum(r["status"] == "PASS" for r in calibration_results))
print(
    "Average score:",
    round(
        sum(r["overall_score"] for r in calibration_results)
        / len(calibration_results),
        2
    )
)

print("\nPosition Bias:")
print("Original winner:", original_result["winner"])
print("Reversed winner:", reversed_result["winner"])
print("Position flip detected:", position_flip)

print("\nSycophancy / Style Bias:")
print("Answer A score:", sycophancy_result["score_A"])
print("Answer B score:", sycophancy_result["score_B"])
print("Winner:", sycophancy_result["winner"])

print("\nStatus: VALIDATION COMPLETED")

LLM JUDGE — FINAL VALIDATION SUMMARY
Judge model: gemini-3.1-flash-lite

Calibration:
Test cases: 1
Passed: 1
Average score: 5.0

Position Bias:
Original winner: Tie
Reversed winner: Tie
Position flip detected: False

Sycophancy / Style Bias:
Answer A score: 5
Answer B score: 1
Winner: A

Status: VALIDATION COMPLETED
